In [ ]:
import os
import sys

ROOT_PATH = '../../'
os.chdir(ROOT_PATH)

sys.path.append(os.path.abspath(ROOT_PATH))

lib_path = os.path.abspath(os.path.join(ROOT_PATH, 'lib'))
if lib_path not in sys.path:
  sys.path.append(lib_path)

In [2]:
import torch
from peft import LoraConfig, TaskType, get_peft_model
from ultralytics import YOLO
from ultralytics.utils.loss import v8SegmentationLoss
from script.training.augmentations import get_augmentation_pipeline
from script.training.rescuenet_dataset import RescueNetDataset
import matplotlib.pyplot as plt

from lib.depth_anything_3.api import DepthAnything3

[WARN ] Dependency `gsplat` is required for rendering 3DGS. Install via: pip install git+https://github.com/nerfstudio-project/gsplat.git@0b4dddf04cb687367602c01196913cde6a743d70


In [11]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

cuda


In [12]:
yolo26 = YOLO("yolo26m-seg.pt", task='segment').to(device)

target_modules = []
for name, module in yolo26.model.model.named_modules():
  if isinstance(module, torch.nn.Conv2d) and module.groups == 1 and not name.startswith('23'):
    target_modules.append(name)

In [13]:
lora_config = LoraConfig(
  r=16,
  lora_alpha=32,
  target_modules=target_modules,
  modules_to_save=["23"],
  bias="none",
  task_type=TaskType.FEATURE_EXTRACTION,
)
peft_model = get_peft_model(yolo26.model, peft_config=lora_config).to(device)
peft_model.print_trainable_parameters()

trainable params: 9,506,296 || all params: 36,618,368 || trainable%: 25.9605


In [14]:
merged_pytorch_model: torch.nn.Module = peft_model.merge_and_unload()
merged_state_dict = merged_pytorch_model.state_dict()

In [15]:
da3_teacher = DepthAnything3.from_pretrained("depth-anything/DA3-LARGE-1.1").to(device)
da3_teacher.eval()

for param in da3_teacher.parameters():
  param.requires_grad = False

lib.depth_anything_3.model.da3
lib.depth_anything_3.model.dinov2.dinov2
[INFO ] using MLP layer as FFN
lib.depth_anything_3.model.dualdpt
lib.depth_anything_3.model.cam_dec
lib.depth_anything_3.model.cam_enc


In [17]:
optimzier = torch.optim.SGD
optimizer = torch.optim.AdamW(yolo26.parameters(), lr=1e-4)

In [18]:
transform = get_augmentation_pipeline()
train_dataset = RescueNetDataset(data_dir='./data/RescueNet/train', transform=transform)
val_dataset = RescueNetDataset(data_dir='./data/RescueNet/val')
test_dataset = RescueNetDataset(data_dir='./data/RescueNet/test')
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=16, shuffle=False)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=16, shuffle=False)
print(f'Train Count: {len(train_dataset)}')
print(f'Validation Count: {len(val_dataset)}')
print(f'Test Count: {len(test_dataset)}')

data\RescueNet\train\train-depth-img
data\RescueNet\val\val-depth-img
data\RescueNet\test\test-depth-img
Train Count: 3595
Validation Count: 449
Test Count: 450


e:\dev\Learning\AI\Pre-Thesis\ai-development\script\training\augmentations.py:16: UserWarning: Argument(s) 'shift_range, scale_range, rotate_range' are not valid for transform Affine
  A.Affine(


In [19]:
for batch_images, batch_targets in train_loader:
  optimizer.zero_grad()
  
  with torch.no_grad():
    da3_depth_features = da3_teacher(batch_images)
  
  yolo_output, yolo_features = yolo26(batch_images)
  
  ultralytics_targets = {
    'bboxes': batch_targets['bboxes'].to(device),
    'cls': batch_targets['cls'].to(device),
    'masks': batch_targets['masks'].to(device),
    'batch_idx': batch_targets['batch_idx'].to(device)
  }
  
  seg_loss = v8SegmentationLoss(yolo_output, ultralytics_targets)
  distill_loss = torch.nn.functional.mse_loss(yolo_features, da3_depth_features)
  
  lambda_weight = 0.5
  loss_total = seg_loss + (lambda_weight * distill_loss)
  
  loss_total.backward()
  optimizer.step()

KeyboardInterrupt: 